# Manual Calibration Linear Models

## tl;dr

- All **8,559 frames** from **nine manual sessions** passed the recorded validity checks; there is one session for each ROI and one camera-control combination.
- The nine-ROI linear classifier reaches **79.7% contact-frame accuracy** under blocked within-session validation.
- The ROI-conditioned force regression remains weak: **MAE 2.53 N**, **RMSE 3.19 N**, and **R² 0.115** when the ROI is known. End-to-end force prediction is slightly worse.
- This is an experimental calibration artifact, not a deployment-ready model. There is no independent repeat manual session for any ROI, so localization generalization cannot yet be measured honestly.


## Context & Methods

The source is `Manual Calibration.zip`, collected August 8, 2026. The model pipeline first assigns one of nine ROI scores, then uses that selected ROI in a single ridge-regularized linear force equation with ROI interaction terms.

### Key Assumptions

- `target_roi_ground_truth` is the manually pressed ROI for the entire session.
- Negative load-cell readings are zero-load noise and are floored to 0 N for fitting.
- ROI `delta_v_mean` is already area-normalized; no cycle normalization is used because these are instantaneous manual frames.
- Five contiguous temporal fifths are used for validation. Training and validation are separated in time, but they still come from the same recording and skin position.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'manual_linear_models').exists():
    ROOT = Path(r'C:\Users\DLSU\OneDrive\Documents\SOFTWARE CALIBRATION\analysis_outputs')
MODEL_DIR = ROOT / 'manual_linear_models'
metrics = json.loads((MODEL_DIR / 'validation_metrics.json').read_text())
model = json.loads((MODEL_DIR / 'manual_linear_models.json').read_text())
per_roi = pd.read_csv(MODEL_DIR / 'per_roi_metrics.csv')
confusion = pd.read_csv(MODEL_DIR / 'localization_confusion.csv', index_col=0)
oof = pd.read_csv(MODEL_DIR / 'out_of_fold_predictions.csv.gz')
metrics['data_quality']


{'raw_sessions': 9, 'raw_frames': 8559, 'valid_model_frames': 8559, 'invalid_or_nonfinite_frames': 0, 'duplicate_session_frame_ids': 0, 'saturation_warning_frames': 0, 'target_session_counts': {'1': 1, '2': 1, '3': 1, '4': 1, '5': 1, '6': 1, '7': 1, '8': 1, '9': 1}, 'camera_control_combinations': [{'applied_exposure': -2.0, 'applied_gain': 0.0, 'applied_white_balance': 4600.0}]}


## Data

### 1. Quality and coverage

In [2]:
pd.Series({
    'sessions': metrics['data_quality']['raw_sessions'],
    'raw frames': metrics['data_quality']['raw_frames'],
    'valid modeling frames': metrics['data_quality']['valid_model_frames'],
    'invalid/nonfinite frames': metrics['data_quality']['invalid_or_nonfinite_frames'],
    'duplicate session-frame IDs': metrics['data_quality']['duplicate_session_frame_ids'],
    'saturation warnings': metrics['data_quality']['saturation_warning_frames'],
    'minimum force (N)': float(oof.force_N.min()),
    'maximum force (N)': float(oof.force_N.max()),
})


sessions                          9.000000
raw frames                     8559.000000
valid modeling frames          8559.000000
invalid/nonfinite frames          0.000000
duplicate session-frame IDs       0.000000
saturation warnings               0.000000
minimum force (N)                 0.000000
maximum force (N)                22.698593
dtype: float64


## Results

### 2. Localization

In [3]:
pd.Series({
    'blocked contact-frame accuracy': metrics['localization_contact_frame_accuracy'],
    'chance accuracy': 1/9,
    'contact frames': int(oof.contact.sum()),
})


blocked contact-frame accuracy       0.796821
chance accuracy                      0.111111
contact frames                    8431.000000
dtype: float64


In [4]:
per_roi[['roi', 'contact_frames', 'localization_accuracy']].set_index('roi')


     contact_frames  localization_accuracy
roi                                       
1              1185               0.789873
2              1070               0.803738
3               931               0.904404
4               947               0.797254
5              1044               0.738506
6               777               0.755470
7               895               0.759777
8               794               0.857683
9               788               0.769036


### 3. Force regression

In [5]:
pd.DataFrame({
    'known ROI': metrics['force_all_frames_known_roi'],
    'end-to-end predicted ROI': metrics['force_all_frames_end_to_end'],
})


          known ROI  end-to-end predicted ROI
frames  8559.000000               8559.000000
mae_N      2.531770                  2.562460
rmse_N     3.194357                  3.243805
r2         0.114741                  0.087121
bias_N     0.028715                 -0.025009


In [6]:
per_roi[['roi', 'force_mae_N_known_roi', 'force_mae_N_end_to_end']].set_index('roi')


     force_mae_N_known_roi  force_mae_N_end_to_end
roi                                               
1                 2.747999                2.773038
2                 3.007882                3.043193
3                 2.220780                2.208689
4                 2.506769                2.531507
5                 2.266389                2.377467
6                 2.526879                2.503482
7                 2.166586                2.180974
8                 2.333332                2.357749
9                 2.615408                2.682668


The manual set improves the optical-force relationship over the automated baseline in R² terms, but an R² of 0.115 still means most frame-to-frame force variation is unexplained. Localization errors add only a small amount of force error; the main bottleneck is the weak optical-to-force relationship itself.

The 79.7% localization number is not directly comparable to a held-out-session score because every ROI has only one recording. Static position/session signatures may therefore inflate it.


## Takeaways

In [7]:
checks = {
    'all source frames retained': metrics['data_quality']['raw_frames'] == metrics['data_quality']['valid_model_frames'],
    'all nine ROI labels present': set(oof.actual_roi.unique()) == set(range(1, 10)),
    'one fold per row': oof.fold.between(0, 4).all(),
    'confusion reconciles to contact rows': int(confusion.to_numpy().sum()) == int(oof.contact.sum()),
    'force predictions finite': np.isfinite(oof.predicted_force_end_to_end_N).all(),
    'model coefficients finite': np.isfinite(model['force_model']['standardized_coefficients']).all(),
}
assert all(checks.values())
checks


{'all source frames retained': True, 'all nine ROI labels present': True, 'one fold per row': np.True_, 'confusion reconciles to contact rows': True, 'force predictions finite': np.True_, 'model coefficients finite': np.True_}


1. The fitted model and inference helper are usable for software integration experiments.
2. Do not use the estimated force for safety control or calibrated reporting.
3. The smallest decisive validation is one additional complete nine-ROI manual run under unchanged camera controls. Keep it untouched until this model is frozen, then report session-held-out force and localization performance.
4. In the application, average five frames before showing the ROI box to reduce flicker, and expose a low-confidence/no-decision state rather than always forcing one of nine boxes.
